In [ ]:
import os
# Remove previous SPARK_HOME if set


# Download and extract Spark 3.5.1
#!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar -xzf spark-3.5.1-bin-hadoop3.tgz
!pip install -q pyspark==3.5.1 delta-spark==3.0.0 findspark

# Set SPARK_HOME and JAVA_HOME
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Install pyspark and delta-spark
#!pip install -q pyspark==3.5.1 delta-spark==2.3.0 findspark # Changed delta-spark to 2.3.0 for compatibility

In [ ]:
# Install Java
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Download Spark from an alternative mirror (works with Colab)
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

# Extract Spark
!tar -xzf spark-3.5.1-bin-hadoop3.tgz

# Install Python libraries
!pip install -q delta-spark findspark


In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("FixVersionMismatch") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [ ]:
import re
from pyspark.sql.functions import col

# Function to clean column names
def clean_column_name(col_name):
    col_name = re.sub(r"[^\w]", "_", col_name)  # Replace non-word chars with "_"
    col_name = re.sub(r"__+", "_", col_name)    # Remove duplicate underscores
    return col_name.strip("_").lower()

# Load the CSV from current directory
df_raw = spark.read.option("header", True).option("inferSchema", True).csv("predictive_maintenance.csv")

# Clean column names
cleaned_columns = [clean_column_name(c) for c in df_raw.columns]
df_raw = df_raw.toDF(*cleaned_columns)

# Preview the cleaned DataFrame
df_raw.show(5)


In [ ]:
from pyspark.sql.functions import col

def clean_column_names(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, col_name.strip().lower().replace(" ", "_"))
    return df

df_bronze = clean_column_names(df_raw)


In [ ]:
bronze_path = "/content/bronze"

df_bronze.write.format("delta").mode("overwrite").save(bronze_path)


In [ ]:
df_bronze_read = spark.read.format("delta").load(bronze_path)
df_bronze_read.show(5)
